In [2]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder
)
from sklearn.compose import ColumnTransformer

df = pd.DataFrame({
    "City": ["Mumbai", "Pune", "Delhi", "Mumbai", "Pune",
             "Delhi", "Mumbai", "Pune", "Delhi", "Mumbai"],
    "Education": ["Graduate", "School", "Postgraduate", "School",
                  "Graduate", "Postgraduate", "Graduate", "School",
                  "Postgraduate", "Graduate"],
    "Membership": ["Gold", "Silver", "Bronze", "Silver", "Gold",
                   "Bronze", "Gold", "Bronze", "Silver", "Gold"],
    "Purchased": ["Yes", "No", "Yes", "No", "Yes",
                  "No", "Yes", "No", "Yes", "No"]
})

X = df.drop(columns="Purchased")
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train)
print(X_test)


     City     Education Membership
6  Mumbai      Graduate       Gold
9  Mumbai      Graduate       Gold
3  Mumbai        School     Silver
1    Pune        School     Silver
2   Delhi  Postgraduate     Bronze
0  Mumbai      Graduate       Gold
4    Pune      Graduate       Gold
5   Delhi  Postgraduate     Bronze
    City     Education Membership
7   Pune        School     Bronze
8  Delhi  Postgraduate     Silver


In [ ]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "nominal",
            OneHotEncoder(
                categories="auto", # discover categories by itself 
                drop='first', # to handle mulicollinearity
                handle_unknown="ignore", # when unknown city or membership comes in training then make all 0
                sparse_output=True, # store only non zero positions
                dtype=int, # encoded output dtype
                min_frequency=None, # cat with less counts gets into separate col
                max_categories=None # keeps only those top n cols with max freq including separate ineff col
            ),
            ["City", "Membership"]
        ),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[[
                    "School", "Graduate", "Postgraduate"
                ]],
                dtype=float,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=float("nan")
            ),
            ["Education"]
        )
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=True
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(preprocessor.get_feature_names_out())
print(X_train_encoded)
print(X_test_encoded)


['nominal__City_Delhi' 'nominal__City_Mumbai' 'nominal__City_Pune'
 'nominal__Membership_Bronze' 'nominal__Membership_Gold'
 'nominal__Membership_Silver' 'ordinal__Education']
[[0. 1. 0. 0. 1. 0. 1.]
 [0. 1. 0. 0. 1. 0. 1.]
 [0. 1. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 1. 0.]
 [1. 0. 0. 1. 0. 0. 2.]
 [0. 1. 0. 0. 1. 0. 1.]
 [0. 0. 1. 0. 1. 0. 1.]
 [1. 0. 0. 1. 0. 0. 2.]]
[[0. 0. 1. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 2.]]


In [5]:
col = "City"

freq_map = X_train[col].value_counts(normalize=True)

X_train["city_freq"] = X_train[col].map(freq_map)
X_test["city_freq"] = X_test[col].map(freq_map).fillna(0)

In [7]:
X_train

,City,Education,Membership,city_freq
6,Mumbai,Graduate,Gold,0.50
9,Mumbai,Graduate,Gold,0.50
3,Mumbai,School,Silver,0.50
1,Pune,School,Silver,0.25
2,Delhi,Postgraduate,Bronze,0.25
0,Mumbai,Graduate,Gold,0.50
4,Pune,Graduate,Gold,0.25
5,Delhi,Postgraduate,Bronze,0.25
